In [1]:
from pathlib import Path
import pandas as pd
import shutil

SOURCE_ROOT = Path(r"C:\static_inference\synthesis_data_40days\data\output\vehicles")
TARGET_ROOT = Path(r"C:\streaming_emulator\data\vehicles")

SIM_FOLDERS = [
    "sim001",
    "sim002",
    "sim003",
    "sim007",
    "sim008",
    "sim009",
    "sim010",
]

EXPECTED_MODULES = {
    "battery",
    "body",
    "engine",
    "transmission",
    "tyre",
}


DROP_COLUMNS = {
    "row_hash",
    "kafka_key",
    "offset",
    "source_file",
    "recon_error_lstm",
    "isolation_score",
    "kde_logp",
    "gmm_logp",
    "composite_score",
    "anomaly_label",
    "anomaly_severity",
    "inference_run_id",
    "inference_timestamp",
    "processing_latency_ms",
    "explain_top_k_json_decoded_json",
    "explain_top_k_json",
    "lstm_window_id",
    "inference_window_id",
    "composite_score_prev",
    "composite_health",
    "timestamp_parsed",
    "com_kms",
}

print("Configuration loaded.")
print("Source root :", SOURCE_ROOT)
print("Target root :", TARGET_ROOT)


Configuration loaded.
Source root : C:\static_inference\synthesis_data_40days\data\output\vehicles
Target root : C:\streaming_emulator\data\vehicles


In [2]:
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"Source root does not exist: {SOURCE_ROOT}")

for sim in SIM_FOLDERS:
    sim_path = SOURCE_ROOT / sim
    if not sim_path.exists():
        raise FileNotFoundError(f"Missing source folder: {sim_path}")

    csvs = list(sim_path.glob("*.csv"))
    if len(csvs) != 5:
        raise RuntimeError(f"{sim} does not contain exactly 5 CSVs")

    found_modules = set()
    for f in csvs:
        for m in EXPECTED_MODULES:
            if m in f.name:
                found_modules.add(m)

    if found_modules != EXPECTED_MODULES:
        raise RuntimeError(
            f"{sim} module mismatch. Found: {found_modules}, Expected: {EXPECTED_MODULES}"
        )

print("Source structure validated.")


Source structure validated.


In [3]:
TARGET_ROOT.mkdir(parents=True, exist_ok=True)

for sim in SIM_FOLDERS:
    target_sim = TARGET_ROOT / sim
    target_sim.mkdir(exist_ok=True)

print("Target folder structure created.")


Target folder structure created.


In [4]:
def process_csv(src_csv: Path, dst_csv: Path):
    df = pd.read_csv(src_csv, low_memory=False)

    cols_to_drop = [c for c in df.columns if c in DROP_COLUMNS]

    df_filtered = df.drop(columns=cols_to_drop)

    df_filtered.to_csv(dst_csv, index=False)

for sim in SIM_FOLDERS:
    src_sim = SOURCE_ROOT / sim
    dst_sim = TARGET_ROOT / sim

    for src_csv in src_sim.glob("*.csv"):
        dst_csv = dst_sim / src_csv.name
        process_csv(src_csv, dst_csv)

    print(f"Processed {sim}")

print("All folders processed successfully.")


Processed sim001
Processed sim002
Processed sim003
Processed sim007
Processed sim008
Processed sim009
Processed sim010
All folders processed successfully.


In [6]:
from pathlib import Path
import pandas as pd


ROOT = Path(r"C:\streaming_emulator\data\vehicles")
DROP_COLS = {"recon_error_dense", "row_idx"}


updated_files = []
skipped_files = []

for csv_path in ROOT.rglob("*.csv"):
    df = pd.read_csv(csv_path, low_memory=False)

    existing = [c for c in df.columns if c in DROP_COLS]

    if not existing:
        skipped_files.append(csv_path.name)
        continue
        
    df.drop(columns=existing, inplace=True)

    df.to_csv(csv_path, index=False)

    updated_files.append({
        "file": csv_path.name,
        "removed_cols": existing,
    })

print(f"Files updated : {len(updated_files)}")
print(f"Files skipped : {len(skipped_files)}")

if updated_files:
    print("\nUpdated files:")
    for u in updated_files:
        print(f"  {u['file']} → removed {u['removed_cols']}")

if skipped_files:
    print("\nSkipped files (columns not present):")
    for f in skipped_files:
        print(f"  {f}")


Files updated : 30
Files skipped : 5

Updated files:
  synthetic_body_inference_scenarioA_sim001.csv → removed ['recon_error_dense']
  synthetic_engine_inference_scenarioA_sim001.csv → removed ['recon_error_dense']
  synthetic_transmission_inference_scenarioA_sim001.csv → removed ['recon_error_dense']
  synthetic_tyre_inference_scenarioA_sim001.csv → removed ['recon_error_dense']
  synthetic_battery_inference_scenarioA_sim002.csv → removed ['row_idx']
  synthetic_body_inference_scenarioA_sim002.csv → removed ['recon_error_dense', 'row_idx']
  synthetic_engine_inference_scenarioA_sim002.csv → removed ['recon_error_dense', 'row_idx']
  synthetic_transmission_inference_scenarioA_sim002.csv → removed ['recon_error_dense', 'row_idx']
  synthetic_tyre_inference_scenarioA_sim002.csv → removed ['recon_error_dense', 'row_idx']
  synthetic_battery_inference_scenarioA_sim003.csv → removed ['row_idx']
  synthetic_body_inference_scenarioA_sim003.csv → removed ['recon_error_dense', 'row_idx']
  synt

In [7]:
from pathlib import Path
import pandas as pd

TARGET_ROOT = Path(r"C:\streaming_emulator\data\vehicles")

SIM_FOLDERS = [
    "sim001",
    "sim002",
    "sim003",
    "sim007",
    "sim008",
    "sim009",
    "sim010",
]

MODULES = [
    "engine",
    "battery",
    "body",
    "transmission",
    "tyre",
]

errors = []

for module in MODULES:
    reference_cols = None
    reference_file = None

    for sim in SIM_FOLDERS:
        sim_path = TARGET_ROOT / sim

        # find module csv
        matches = [f for f in sim_path.glob("*.csv") if module in f.name]
        if len(matches) != 1:
            raise RuntimeError(
                f"{sim}: expected exactly one CSV for module '{module}', found {len(matches)}"
            )

        csv_path = matches[0]

        # read only header (fast + safe)
        cols = list(pd.read_csv(csv_path, nrows=0).columns)

        if reference_cols is None:
            reference_cols = cols
            reference_file = csv_path
        else:
            if cols != reference_cols:
                errors.append({
                    "module": module,
                    "sim": sim,
                    "file": csv_path.name,
                    "reference_file": reference_file.name,
                    "reference_cols": reference_cols,
                    "current_cols": cols,
                })

if errors:
    print("❌ SCHEMA MISMATCH DETECTED\n")
    for e in errors:
        print(f"Module        : {e['module']}")
        print(f"Sim           : {e['sim']}")
        print(f"File          : {e['file']}")
        print(f"Reference     : {e['reference_file']}")
        print(f"Reference cols: {e['reference_cols']}")
        print(f"Current cols  : {e['current_cols']}")
        print("-" * 80)
    raise RuntimeError("Module-wise schema consistency check failed.")
else:
    print("✅ All module CSVs have identical columns across all sims.")


✅ All module CSVs have identical columns across all sims.


In [9]:
from pathlib import Path
import pandas as pd
import json
from collections import OrderedDict

# =========================
# CONFIG
# =========================

ROOT = Path(r"C:\streaming_emulator\data\vehicles")

MODULES = [
    "engine",
    "battery",
    "body",
    "transmission",
    "tyre",
]

OUT_JSON = ROOT / "modules_schema_master.json"

sim_dirs = sorted([p for p in ROOT.iterdir() if p.is_dir()])
if not sim_dirs:
    raise RuntimeError("No sim folders found under vehicles root.")

reference_sim = sim_dirs[0]
print(f"Using reference sim: {reference_sim.name}")


master_schema = OrderedDict()
master_schema["schema_type"] = "vehicle_module_master_schema"
master_schema["reference_sim"] = reference_sim.name
master_schema["modules"] = OrderedDict()

for module in MODULES:
    matches = [f for f in reference_sim.glob("*.csv") if module in f.name]
    if len(matches) != 1:
        raise RuntimeError(
            f"Reference sim '{reference_sim.name}' has {len(matches)} files for module '{module}'"
        )

    csv_path = matches[0]
    df = pd.read_csv(csv_path, low_memory=False)

    cols_info = OrderedDict()
    for col in df.columns:
        cols_info[col] = {
            "dtype": str(df[col].dtype),
            "nullable": bool(df[col].isna().any()),
            "example_value": None if df[col].dropna().empty else df[col].dropna().iloc[0],
        }

    master_schema["modules"][module] = {
        "file_pattern": f"*{module}*.csv",
        "column_count": len(df.columns),
        "columns": cols_info,
    }

with OUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(master_schema, f, indent=2, ensure_ascii=False)

print(f"Master schema written → {OUT_JSON}")


Using reference sim: sim001
Master schema written → C:\streaming_emulator\data\vehicles\modules_schema_master.json
